# US Consumption Tax vs Income Tax Reform Base Updater

This notebook reconstructs Appendix Table 1 based on the methodology used in the Tax Foundation's paper, "US Consumption Tax vs Income Tax". It constructs a cash-flow VAT base which roughly matches total domestic consumption by mapping the row components directly to the Bureau of Economic Analysis (BEA) National Income and Product Accounts (NIPA) tables.

It automatically fetches the latest data via the St. Louis FRED API.

In [1]:
import urllib.request
import csv
import pandas as pd

def get_annual_average(series_id, target_year):
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    try:
        html = urllib.request.urlopen(req).read().decode('utf-8')
        reader = csv.reader(html.strip().split('\n'))
        next(reader) 
        values = []
        for row in reader:
            if len(row) > 1 and row[1] != '.' and row[0].startswith(str(target_year)):
                values.append(float(row[1]))
        return sum(values) / len(values) if values else 0
    except Exception as e:
        print(f'Error fetching {series_id}: {e}')
        return 0

In [9]:
def generate_table(year):
    coe = get_annual_average('COE', year)
    cprofit = get_annual_average('CPROFIT', year)
    propinc = get_annual_average('PROPINC', year)
    netint = get_annual_average('B471RC1Q027SBEA', year) 
    cfc = get_annual_average('COFC', year)
    taxes_pi = get_annual_average('GDITAXES', year) 
    investment = get_annual_average('GPDI', year)
    
    netcap = cprofit + propinc + netint + cfc + taxes_pi - investment
    imports = get_annual_average('IMPGS', year)
    exports = get_annual_average('EXPGS', year)
    net_imports = imports - exports
    hh_inv = get_annual_average('W988RC1A027NBEA', year)
    hh_flow_inc = get_annual_average('B952RC1A027NBEA', year)
    housing_adj = hh_inv - hh_flow_inc
    
    noncomp_rate = 0.075
    broad_base_before = coe + netcap + net_imports + housing_adj
    non_compliance = -noncomp_rate * broad_base_before
    broad_vat_base = broad_base_before + non_compliance
    gdp = get_annual_average('GDP', year)
    
    data = [
        {'Component': 'Compensation of Employees', 'Billions': coe},
        {'Component': 'Net Capital Income (Profits+Interest+CFC+Excise-Investment)', 'Billions': netcap},
        {'Component': 'Net Imports', 'Billions': net_imports},
        {'Component': 'Housing Adjustment (Household Investment Less Net Housing Value Added)', 'Billions': housing_adj},
        {'Component': f'Non-Compliance ({noncomp_rate:.0%} rate)', 'Billions': non_compliance},
        {'Component': 'Broad VAT Base (Before Exemptions)', 'Billions': broad_vat_base}
    ]
    
    df = pd.DataFrame(data)
    df['% of GDP'] = (df['Billions'] / gdp) * 100
    
    # Format the columns for display
    df['Billions ($)'] = df['Billions'].apply(lambda x: f'${x:,.1f}')
    df['% of GDP'] = df['% of GDP'].apply(lambda x: f'{x:.1f}%')
    
    # Drop the unformatted column and rearrange
    df = df[['Component', 'Billions ($)', '% of GDP']]
    
    # Print Output
    print(f'\n======== APPENDIX TABLE 1 (Updated {year}) ========')
    print(f'Total GDP for {year}: ${gdp:,.1f} Billion\n')
    return df

In [10]:
# Generate and display the table for 2024
df_2024 = generate_table('2024')
df_2024


======== APPENDIX TABLE 1 (Updated 2024) ========
Total GDP for 2024: $29,298.0 Billion



,Component,Billions ($),% of GDP
0,Compensation of Employees,"$15,027.1",51.3%
1,Net Capital Income (Profits+Interest+CFC+Excis...,"$7,516.2",25.7%
2,Net Imports,$898.5,3.1%
3,Housing Adjustment (Household Investment Less ...,$-676.9,-2.3%
4,Non-Compliance (8% rate),"$-1,707.4",-5.8%
5,Broad VAT Base (Before Exemptions),"$21,057.5",71.9%
